# Baseline Movement Disorder Classifiers

LRNG: National (India) | CC: Human Values | SDG: 3 | Data: Time series | Thrust: Intelligent Edge Technologies | Screening aid only

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score, precision_score, recall_score, roc_auc_score

np.random.seed(42)
sys.path.append("..")

from data_loading.unified_loader import build_pads_primary_100hz_dataset, build_uea_within_source_dataset
from preprocessing.features import extract_features_dataset
from models.baselines import evaluate_fft_threshold_rule, evaluate_rms_threshold_rule

Loading Track 1 primary dataset: PADS 100 Hz, 6 channels, 5 tremor tasks, first 2 s trimmed

In [2]:
pads_root = os.path.join("..", "data", "pads parkinsons-disease smartwatch dataset-1.0.0")
windows_100hz, meta_100hz = build_pads_primary_100hz_dataset(pads_root, tasks=None, trim_start_samples=200)

os.makedirs(os.path.join("..", "results"), exist_ok=True)
np.savez_compressed(os.path.join("..", "results", "pads_primary_100hz.npz"), windows=windows_100hz)
meta_100hz.to_csv(os.path.join("..", "results", "pads_primary_100hz_metadata.csv"), index=False)

print("PADS 100 Hz windows shape:", windows_100hz.shape)
print("Cohort label distribution:")
print(meta_100hz["cohort_label"].value_counts())
print("Unique subjects count:", meta_100hz["subject_id"].nunique())

features_100hz = extract_features_dataset(windows_100hz, sampling_rate=100.0)
print("Handcrafted features shape:", features_100hz.shape)
np.save(os.path.join("..", "results", "pads_features_100hz.npy"), features_100hz)

PADS 100 Hz windows shape: (25500, 300, 6)
Cohort label distribution:
cohort_label
Parkinsons-like    18224
Normal              5372
ET-like             1904
Name: count, dtype: int64
Unique subjects count: 375


Handcrafted features shape: (25500, 30)


Track 1 Random Forest baseline across 3 fold seeds (5 folds each) with subject bootstrap CIs and AUROC

In [3]:
y_track1 = meta_100hz["cohort_label"].map({"Normal": 0, "Parkinsons-like": 1, "ET-like": 2}).values
groups_track1 = meta_100hz["subject_id"].values
target_names = ["Normal", "Parkinsons-like", "ET-like"]
unique_subjects = meta_100hz["subject_id"].unique()
sub_true = np.array([y_track1[groups_track1 == s][0] for s in unique_subjects])

split_seeds = [42, 43, 44]
save_splits = {}
seed_sub_f1 = []
seed_sub_bacc = []
seed_sub_probs = []
seed_win_probs = []

for s in split_seeds:
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=s)
    oof_p = np.zeros((len(y_track1), 3))
    
    for fold, (tr, te) in enumerate(sgkf.split(features_100hz, y_track1, groups_track1)):
        save_splits[f"split_{s}_tr_{fold}"] = tr
        save_splits[f"split_{s}_te_{fold}"] = te
        clf = RandomForestClassifier(n_estimators=100, max_depth=15, class_weight="balanced", random_state=s, n_jobs=-1)
        clf.fit(features_100hz[tr], y_track1[tr])
        oof_p[te] = clf.predict_proba(features_100hz[te])
        
    sub_p = np.array([np.mean(oof_p[groups_track1 == sub], axis=0) for sub in unique_subjects])
    sub_pred = np.argmax(sub_p, axis=1)
    s_f1 = f1_score(sub_true, sub_pred, average="macro")
    s_bacc = balanced_accuracy_score(sub_true, sub_pred)
    seed_sub_f1.append(s_f1)
    seed_sub_bacc.append(s_bacc)
    seed_sub_probs.append(sub_p)
    seed_win_probs.append(oof_p)
    print(f"Split seed {s}: Subject Macro-F1 = {s_f1:.4f}, Subject Balanced Acc = {s_bacc:.4f}")

np.savez(os.path.join("..", "results", "fold_splits_3seeds.npz"), **save_splits)
print("Saved 3 split seeds (15 folds total) to results/fold_splits_3seeds.npz")

avg_sub_probs = np.mean(seed_sub_probs, axis=0)
final_sub_preds = np.argmax(avg_sub_probs, axis=1)
avg_win_probs = np.mean(seed_win_probs, axis=0)
final_win_preds = np.argmax(avg_win_probs, axis=1)

sub_macro_f1 = float(f1_score(sub_true, final_sub_preds, average="macro"))
sub_bacc = float(balanced_accuracy_score(sub_true, final_sub_preds))
win_macro_f1 = float(f1_score(y_track1, final_win_preds, average="macro"))
win_bacc = float(balanced_accuracy_score(y_track1, final_win_preds))

auc_normal = float(roc_auc_score(sub_true == 0, avg_sub_probs[:, 0]))
auc_pd = float(roc_auc_score(sub_true == 1, avg_sub_probs[:, 1]))
auc_et = float(roc_auc_score(sub_true == 2, avg_sub_probs[:, 2]))
auc_macro = float((auc_normal + auc_pd + auc_et) / 3.0)

def compute_bootstrap_ci(y_t, probs, metric_fn, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    scores = []
    n = len(y_t)
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        scores.append(metric_fn(y_t[idx], probs[idx]))
    return float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))

f1_ci = compute_bootstrap_ci(sub_true, avg_sub_probs, lambda yt, p: f1_score(yt, np.argmax(p, axis=1), average="macro"))
bacc_ci = compute_bootstrap_ci(sub_true, avg_sub_probs, lambda yt, p: balanced_accuracy_score(yt, np.argmax(p, axis=1)))

print("\n=== Track 1 Random Forest Baseline Seed-Averaged Results ===")
print(f"Subject Macro-F1 across 3 split seeds: {np.mean(seed_sub_f1):.4f} +- {np.std(seed_sub_f1):.4f}")
print(f"Pooled Subject-Level Macro-F1: {sub_macro_f1:.4f} [95% Bootstrap CI: {f1_ci[0]:.4f} - {f1_ci[1]:.4f}]")
print(f"Pooled Subject-Level Balanced Acc: {sub_bacc:.4f} [95% Bootstrap CI: {bacc_ci[0]:.4f} - {bacc_ci[1]:.4f}]")
print(f"Subject One-vs-Rest AUROC: Normal={auc_normal:.4f}, PD={auc_pd:.4f}, ET={auc_et:.4f}, Macro={auc_macro:.4f}")
print(f"Window-Level Macro-F1: {win_macro_f1:.4f} | Balanced Acc: {win_bacc:.4f}")

sub_cm = confusion_matrix(sub_true, final_sub_preds)
print("\nSubject-Level Confusion Matrix:")
print(pd.DataFrame(sub_cm, index=[f"True_{c}" for c in target_names], columns=[f"Pred_{c}" for c in target_names]))
rec_per_class = sub_cm.diagonal() / sub_cm.sum(axis=1)
for c, r in zip(target_names, rec_per_class):
    print(f"Recall for {c}: {r:.4f}")

track1_rf_records = [{
    "Model": "Random Forest (Baseline)",
    "Sampling_Rate": "100 Hz",
    "Channels": 6,
    "Subject_Macro_F1": sub_macro_f1,
    "Subject_F1_CI_Low": f1_ci[0],
    "Subject_F1_CI_High": f1_ci[1],
    "Subject_Balanced_Acc": sub_bacc,
    "Subject_BAcc_CI_Low": bacc_ci[0],
    "Subject_BAcc_CI_High": bacc_ci[1],
    "AUROC_Normal": auc_normal,
    "AUROC_PD": auc_pd,
    "AUROC_ET": auc_et,
    "AUROC_Macro": auc_macro,
    "Window_Macro_F1": win_macro_f1,
    "Window_Balanced_Acc": win_bacc
}]
os.makedirs(os.path.join("..", "results", "metrics"), exist_ok=True)
pd.DataFrame(track1_rf_records).to_csv(os.path.join("..", "results", "metrics", "track1_baselines.csv"), index=False)

Split seed 42: Subject Macro-F1 = 0.4221, Subject Balanced Acc = 0.4162


Split seed 43: Subject Macro-F1 = 0.3741, Subject Balanced Acc = 0.3809


Split seed 44: Subject Macro-F1 = 0.4135, Subject Balanced Acc = 0.4085
Saved 3 split seeds (15 folds total) to results/fold_splits_3seeds.npz



=== Track 1 Random Forest Baseline Seed-Averaged Results ===
Subject Macro-F1 across 3 split seeds: 0.4032 +- 0.0209
Pooled Subject-Level Macro-F1: 0.3890 [95% Bootstrap CI: 0.3250 - 0.4534]
Pooled Subject-Level Balanced Acc: 0.3924 [95% Bootstrap CI: 0.3513 - 0.4370]
Subject One-vs-Rest AUROC: Normal=0.8583, PD=0.7812, ET=0.8925, Macro=0.8440
Window-Level Macro-F1: 0.4653 | Balanced Acc: 0.4509

Subject-Level Confusion Matrix:
                      Pred_Normal  Pred_Parkinsons-like  Pred_ET-like
True_Normal                     7                    72             0
True_Parkinsons-like            0                   263             5
True_ET-like                    0                    25             3
Recall for Normal: 0.0886
Recall for Parkinsons-like: 0.9813
Recall for ET-like: 0.1071


Track 1 FFT Threshold Rule baseline for binary tremor screening (Healthy vs Tremor)

In [4]:
y_binary = (y_track1 > 0).astype(int)
fft_preds = evaluate_fft_threshold_rule(features_100hz, dom_freq_col=1, band_pwr_col=2)
fft_f1 = f1_score(y_binary, fft_preds, average="macro")
fft_rec = recall_score(y_binary, fft_preds, pos_label=1)
fft_spec = recall_score(y_binary, fft_preds, pos_label=0)

print("=== Track 1 FFT Threshold Rule Baseline ===")
print(f"Binary Tremor Detection Macro-F1: {fft_f1:.4f}")
print(f"Sensitivity (Tremor Recall): {fft_rec:.4f}")
print(f"Specificity (Healthy Recall): {fft_spec:.4f}")

=== Track 1 FFT Threshold Rule Baseline ===
Binary Tremor Detection Macro-F1: 0.5229
Sensitivity (Tremor Recall): 0.5830
Specificity (Healthy Recall): 0.5624


Track 2 Baseline: UEA Epilepsy Seizure Detection with Train-Fitted RMS Threshold

In [5]:
uea_dir = os.path.join("..", "data", "Epilepsy")
uea_windows, uea_meta = build_uea_within_source_dataset(uea_dir)

np.savez_compressed(os.path.join("..", "results", "uea_within_source_16hz.npz"), windows=uea_windows)
uea_meta.to_csv(os.path.join("..", "results", "uea_within_source_metadata.csv"), index=False)

uea_features = extract_features_dataset(uea_windows, sampling_rate=16.0)
np.save(os.path.join("..", "results", "uea_features_16hz.npy"), uea_features)
y_uea = (uea_meta["cohort_label"] == "Seizure-like (simulated)").astype(int).values

train_mask = (uea_meta["split"] == "train").values
test_mask = (uea_meta["split"] == "test").values

train_series_ids = uea_meta.loc[train_mask, "series_id"].unique()
train_series_true = np.array([y_uea[uea_meta["series_id"] == sid][0] for sid in train_series_ids])
train_series_rms = np.array([np.max(uea_features[uea_meta["series_id"] == sid, 0]) for sid in train_series_ids])

test_series_ids = uea_meta.loc[test_mask, "series_id"].unique()
test_series_true = np.array([y_uea[uea_meta["series_id"] == sid][0] for sid in test_series_ids])
test_series_rms = np.array([np.max(uea_features[uea_meta["series_id"] == sid, 0]) for sid in test_series_ids])
total_test_seizures = int(np.sum(test_series_true == 1))

best_thresh = 0.0
best_train_f1 = 0.0
for th in np.linspace(float(np.min(train_series_rms)), float(np.max(train_series_rms)), 200):
    preds = (train_series_rms >= th).astype(int)
    f1 = float(f1_score(train_series_true, preds, average="macro"))
    if f1 > best_train_f1:
        best_train_f1 = f1
        best_thresh = float(th)

print(f"Selected Train RMS Threshold: {best_thresh:.4f} (Train Macro-F1: {best_train_f1:.4f})")

test_preds_rms = (test_series_rms >= best_thresh).astype(int)
detected_rms = int(np.sum((test_series_true == 1) & (test_preds_rms == 1)))
rec_rms = float(recall_score(test_series_true, test_preds_rms, pos_label=1))
prec_rms = float(precision_score(test_series_true, test_preds_rms, pos_label=1))
f1_rms = float(f1_score(test_series_true, test_preds_rms, average="macro"))

rf_uea = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1)
rf_uea.fit(uea_features[train_mask], y_uea[train_mask])
test_probs_rf = rf_uea.predict_proba(uea_features[test_mask])[:, 1]

series_preds_rf = np.array([int(np.mean(test_probs_rf[(uea_meta.loc[test_mask, "series_id"] == sid).values]) >= 0.5) for sid in test_series_ids])
detected_rf = int(np.sum((test_series_true == 1) & (series_preds_rf == 1)))
f1_rf = float(f1_score(test_series_true, series_preds_rf, average="macro"))
rec_rf = float(recall_score(test_series_true, series_preds_rf, pos_label=1))
prec_rf = float(precision_score(test_series_true, series_preds_rf, pos_label=1))

print("=== Track 2 Test Set Baseline Results ===")
print(f"RMS Threshold Rule (Train-Fitted): Detected {detected_rms} of {total_test_seizures} seizures | Recall: {rec_rms:.4f} | Precision: {prec_rms:.4f} | Macro-F1: {f1_rms:.4f}")
print(f"Random Forest: Detected {detected_rf} of {total_test_seizures} seizures | Recall: {rec_rf:.4f} | Precision: {prec_rf:.4f} | Macro-F1: {f1_rf:.4f}")

track2_records = [
    {"Model": "Random Forest (Baseline)", "Series_Detected": f"{detected_rf} of {total_test_seizures}", "Seizure_Recall": rec_rf, "Seizure_Precision": prec_rf, "Macro_F1": f1_rf},
    {"Model": "RMS Threshold Rule (Train-Fitted)", "Series_Detected": f"{detected_rms} of {total_test_seizures}", "Seizure_Recall": rec_rms, "Seizure_Precision": prec_rms, "Macro_F1": f1_rms}
]
pd.DataFrame(track2_records).to_csv(os.path.join("..", "results", "metrics", "track2_baselines.csv"), index=False)

Selected Train RMS Threshold: 5.6683 (Train Macro-F1: 0.6566)


=== Track 2 Test Set Baseline Results ===
RMS Threshold Rule (Train-Fitted): Detected 16 of 34 seizures | Recall: 0.4706 | Precision: 0.5714 | Macro-F1: 0.6880
Random Forest: Detected 26 of 34 seizures | Recall: 0.7647 | Precision: 1.0000 | Macro-F1: 0.9148


Summary of Baseline Benchmark Results

In [6]:
print("Track 1 Baseline Summary:")
print(pd.DataFrame(track1_rf_records).to_string(index=False))
print("\nTrack 2 Baseline Summary:")
print(pd.DataFrame(track2_records).to_string(index=False))

Track 1 Baseline Summary:
                   Model Sampling_Rate  Channels  Subject_Macro_F1  Subject_F1_CI_Low  Subject_F1_CI_High  Subject_Balanced_Acc  Subject_BAcc_CI_Low  Subject_BAcc_CI_High  AUROC_Normal  AUROC_PD  AUROC_ET  AUROC_Macro  Window_Macro_F1  Window_Balanced_Acc
Random Forest (Baseline)        100 Hz         6          0.389012           0.324987            0.453426              0.392365             0.351302              0.437048      0.858279  0.781246  0.892548     0.844024          0.46528             0.450854

Track 2 Baseline Summary:
                            Model Series_Detected  Seizure_Recall  Seizure_Precision  Macro_F1
         Random Forest (Baseline)        26 of 34        0.764706           1.000000  0.914815
RMS Threshold Rule (Train-Fitted)        16 of 34        0.470588           0.571429  0.687971
